In [ ]:
import os
import GEOparse
import pandas as pd

print("[*] Loading GSE65682 into memory (This is a big one, give it 2-3 minutes)...")
gse = GEOparse.get_GEO(filepath="/workspace/data/raw/GSE65682_family.soft.gz")

# ---------------------------------------------------------
# 1. CLEAN THE CLINICAL DATA
# ---------------------------------------------------------
clinical_df = gse.phenotype_data

# Grab only the characteristic columns
char_cols = [col for col in clinical_df.columns if 'characteristics' in col]
clean_clinical = clinical_df[char_cols].copy()

# Rename the columns to remove 'characteristics_ch1.X.'
# e.g., 'characteristics_ch1.6.mortality_event_28days' -> 'mortality_event_28days'
clean_clinical.columns = [col.split('.')[-1] for col in clean_clinical.columns]

# Filter out the healthy controls (where mortality is 'NA' or missing)
clean_clinical = clean_clinical[clean_clinical['mortality_event_28days'] != 'NA']
clean_clinical = clean_clinical.dropna(subset=['mortality_event_28days'])

# Convert mortality and age to numeric values
clean_clinical['mortality_event_28days'] = pd.to_numeric(clean_clinical['mortality_event_28days'])
clean_clinical['age'] = pd.to_numeric(clean_clinical['age'], errors='coerce')

print(f"\n[+] Cleaned Clinical Data:")
print(f"    Total Septic Patients: {clean_clinical.shape[0]}")
print(f"    Survivors: {sum(clean_clinical['mortality_event_28days'] == 0)}")
print(f"    Non-Survivors: {sum(clean_clinical['mortality_event_28days'] == 1)}")

# Save to processed folder
processed_dir = "/workspace/data/processed"
clinical_file = os.path.join(processed_dir, "GSE65682_clinical_clean.csv")
clean_clinical.to_csv(clinical_file)

# ---------------------------------------------------------
# 2. EXTRACT THE GENE EXPRESSION MATRIX
# ---------------------------------------------------------
print("\n[*] Pivoting gene expression matrix (Takes ~5 minutes due to 800+ patients)...")
expr_df = gse.pivot_samples('VALUE')

# Ensure we only keep the genes for the patients who are in our cleaned clinical set
expr_df = expr_df[clean_clinical.index]

expr_file = os.path.join(processed_dir, "GSE65682_expression.csv")
expr_df.to_csv(expr_file)

print(f"[+] Saved Expression Data: {expr_df.shape[0]} genes, {expr_df.shape[1]} patients -> {expr_file}")
print("\n=== PHASE 2: DATA ENGINEERING COMPLETE ===")

27-Apr-2026 16:58:31 INFO GEOparse - Parsing /workspace/data/raw/GSE65682_family.soft.gz: 
27-Apr-2026 16:58:31 DEBUG GEOparse - DATABASE: GeoMiame
27-Apr-2026 16:58:31 DEBUG GEOparse - SERIES: GSE65682
27-Apr-2026 16:58:31 DEBUG GEOparse - PLATFORM: GPL13667


[*] Loading GSE65682 into memory (This is a big one, give it 2-3 minutes)...


/opt/conda/lib/python3.10/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
27-Apr-2026 16:58:37 DEBUG GEOparse - SAMPLE: GSM1602801
27-Apr-2026 16:58:37 DEBUG GEOparse - SAMPLE: GSM1602802
27-Apr-2026 16:58:37 DEBUG GEOparse - SAMPLE: GSM1602803
27-Apr-2026 16:58:37 DEBUG GEOparse - SAMPLE: GSM1602804
27-Apr-2026 16:58:38 DEBUG GEOparse - SAMPLE: GSM1602805
27-Apr-2026 16:58:38 DEBUG GEOparse - SAMPLE: GSM1602806
27-Apr-2026 16:58:38 DEBUG GEOparse - SAMPLE: GSM1602807
27-Apr-2026 16:58:38 DEBUG GEOparse - SAMPLE: GSM1602808
27-Apr-2026 16:58:38 DEBUG GEOparse - SAMPLE: GSM1602809
27-Apr-2026 16:58:38 DEBUG GEOparse - SAMPLE: GSM1602810
27-Apr-2026 16:58:38 DEBUG GEOparse - SAMPLE: GSM1602811
27-Apr-2026 16:58:38 DEBUG GEOparse - SAMPLE: GSM1602812
27-Apr-2026 16:58:38 DEBUG GEOparse - SAMPLE: GSM1602813
27-Apr-2026 16:58:38 DEBUG GEOpa


[+] Cleaned Clinical Data:
    Total Septic Patients: 479
    Survivors: 365
    Non-Survivors: 114

[*] Pivoting gene expression matrix (Takes ~5 minutes due to 800+ patients)...
